# Late Fusion Implementation


In [1]:
!unzip /content/RGB_P.zip

Archive:  /content/RGB_P.zip
   creating: RGB_P/
   creating: RGB_P/Patches/
   creating: RGB_P/Patches/Abnormal(Ulcer)/
  inflating: RGB_P/Patches/Abnormal(Ulcer)/1.jpg  
  inflating: RGB_P/Patches/Abnormal(Ulcer)/10.jpg  
  inflating: RGB_P/Patches/Abnormal(Ulcer)/100.jpg  
  inflating: RGB_P/Patches/Abnormal(Ulcer)/101.jpg  
  inflating: RGB_P/Patches/Abnormal(Ulcer)/102.jpg  
  inflating: RGB_P/Patches/Abnormal(Ulcer)/103.jpg  
  inflating: RGB_P/Patches/Abnormal(Ulcer)/104.jpg  
  inflating: RGB_P/Patches/Abnormal(Ulcer)/105.jpg  
  inflating: RGB_P/Patches/Abnormal(Ulcer)/106.jpg  
  inflating: RGB_P/Patches/Abnormal(Ulcer)/107.jpg  
  inflating: RGB_P/Patches/Abnormal(Ulcer)/108.jpg  
  inflating: RGB_P/Patches/Abnormal(Ulcer)/109.jpg  
  inflating: RGB_P/Patches/Abnormal(Ulcer)/11.jpg  
  inflating: RGB_P/Patches/Abnormal(Ulcer)/110.jpg  
  inflating: RGB_P/Patches/Abnormal(Ulcer)/111.jpg  
  inflating: RGB_P/Patches/Abnormal(Ulcer)/112.jpg  
  inflating: RGB_P/Patches/Abnormal

In [2]:
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torchvision import transforms, models
from PIL import Image
import joblib
from sklearn.metrics import accuracy_score, classification_report, roc_auc_score

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {DEVICE}")

Device: cpu


## 1. Load frozen models

Adjust paths to wherever you saved each model in Phase 3 (hyperparameter tuning).

In [8]:
# --- RGB: EfficientNet-B3 ---
IMG_SIZE = 224
NUM_CLASSES = 2

rgb_model = models.efficientnet_b3(weights=None)
in_features = rgb_model.classifier[1].in_features
rgb_model.classifier = nn.Sequential(
    nn.Dropout(p=0.4, inplace=True),
    nn.Linear(in_features, 256),
    nn.ReLU(),
    nn.Dropout(p=0.3),
    nn.Linear(256, NUM_CLASSES),
)
rgb_model.load_state_dict(torch.load("/content/best_efficientnet_b3.pth", map_location=DEVICE))
rgb_model.to(DEVICE).eval()

IMAGENET_MEAN = [0.485, 0.456, 0.406]
IMAGENET_STD = [0.229, 0.224, 0.225]
rgb_transform = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize(mean=IMAGENET_MEAN, std=IMAGENET_STD),
])

# --- Thermal: Random Forest ---
rf_model = joblib.load("/content/best_rf_thermal.joblib")
THERMAL_FEATURE_COLS = [
    "mean_temp", "max_temp", "std_temp", "range_temp",
    "hot_spot_count", "temp_gradient",
    "glcm_contrast", "glcm_homogeneity", "glcm_entropy",
]

# --- Plantar: XGBoost ---
xgb_model = joblib.load("/content/best_xgb_plantar.joblib")
PLANTAR_FEATURE_COLS = [
    "mean_pressure", "max_pressure", "contact_area", "cop_x", "cop_y",
    "forefoot_ratio", "midfoot_ratio", "hindfoot_ratio",
    "glcm_contrast", "glcm_homogeneity", "glcm_entropy",
]

print("All three models loaded.")

All three models loaded.


## 2. Load paired data + source feature tables

In [9]:
thermal_df_raw = pd.read_csv("/content/thermal_features.csv")
thermal_df_raw["split"] = thermal_df_raw["split"].replace({"val": "holdout"})
plantar_df_raw = pd.read_csv("/content/plantar_features.csv")
plantar_df_raw["split"] = plantar_df_raw["split"].replace({"test": "holdout"})

# thermal_row_idx / plantar_row_idx in the paired CSVs are positions within the
# split-filtered, reset-index tables from CrossModal_Pairing.ipynb -- must rebuild
# the same split-filtered, reset-index tables here before using .loc on them.
thermal_by_split = {s: thermal_df_raw[thermal_df_raw["split"] == s].reset_index(drop=True) for s in ["train", "holdout"]}
plantar_by_split = {s: plantar_df_raw[plantar_df_raw["split"] == s].reset_index(drop=True) for s in ["train", "holdout"]}

paired_train = pd.read_csv("/content/paired_train.csv")
paired_holdout = pd.read_csv("/content/paired_holdout.csv")

print(f"Train pairs: {len(paired_train)} | Holdout pairs: {len(paired_holdout)}")

Train pairs: 1196 | Holdout pairs: 299


## 3. Per-modality probability extraction

- `P_Thermal_risk = P(DM)`
- `P_Plantar_risk = P(flat foot) + P(over-arch)`  (normal arch = low risk)

In [10]:
def rgb_probs_batch(paths, batch_size=32):
    probs = []
    with torch.no_grad():
        for i in range(0, len(paths), batch_size):
            batch_paths = paths[i:i + batch_size]
            imgs = torch.stack([
                rgb_transform(Image.open(p).convert("RGB")) for p in batch_paths
            ]).to(DEVICE)
            out = rgb_model(imgs)
            p_dfu = torch.softmax(out, dim=1)[:, 1].cpu().numpy()
            probs.extend(p_dfu.tolist())
    return np.array(probs)


def thermal_risk_batch(row_idxs, split_name):
    feats = thermal_by_split[split_name].loc[row_idxs, THERMAL_FEATURE_COLS].values
    classes = list(rf_model.classes_)
    dm_idx = classes.index(1) if 1 in classes else 1
    return rf_model.predict_proba(feats)[:, dm_idx]


def plantar_risk_batch(row_idxs, split_name):
    feats = plantar_by_split[split_name].loc[row_idxs, PLANTAR_FEATURE_COLS].values
    probs = xgb_model.predict_proba(feats)
    # LabelEncoder fit order was ['flat foot','normal','over-arch'] -> indices 0,1,2
    risk = probs[:, 0] + probs[:, 2]  # flat foot + over-arch
    return risk

In [11]:
print("RF classes (thermal):", rf_model.classes_)
print("XGB classes (plantar):", xgb_model.classes_)
# If these don't match [0,1] -> ['Control','DM'] and [0,1,2] -> ['flat foot','normal','over-arch'],
# fix the index logic in the two functions above before proceeding.

RF classes (thermal): [0 1]
XGB classes (plantar): [0 1 2]


## 4. Run inference on both splits, save probability tables

In [12]:
def run_inference(paired_df, split_name):
    out = paired_df.copy()
    out["p_rgb"] = rgb_probs_batch(out["rgb_path"].tolist())
    out["p_thermal_risk"] = thermal_risk_batch(out["thermal_row_idx"].tolist(), split_name)
    out["p_plantar_risk"] = plantar_risk_batch(out["plantar_row_idx"].tolist(), split_name)
    return out


train_probs = run_inference(paired_train, "train")
holdout_probs = run_inference(paired_holdout, "holdout")

train_probs.to_csv("/content/fusion_train_probs.csv", index=False)
holdout_probs.to_csv("/content/fusion_holdout_probs.csv", index=False)

print(f"Saved fusion_train_probs.csv ({len(train_probs)} rows)")
print(f"Saved fusion_holdout_probs.csv ({len(holdout_probs)} rows)")

Saved fusion_train_probs.csv (1196 rows)
Saved fusion_holdout_probs.csv (299 rows)


## 5. Placeholder fusion (equal weights) + single-modality baselines


In [13]:
def evaluate(y_true, y_prob, name):
    y_pred = (y_prob >= 0.5).astype(int)
    acc = accuracy_score(y_true, y_pred)
    auc = roc_auc_score(y_true, y_prob)
    print(f"\n--- {name} ---")
    print(f"Accuracy: {acc*100:.2f}% | AUC: {auc:.4f}")
    print(classification_report(y_true, y_pred, target_names=["No DFU", "DFU"]))
    return acc, auc


w1, w2, w3 = 1 / 3, 1 / 3, 1 / 3
holdout_probs["p_fused"] = (
    w1 * holdout_probs["p_rgb"]
    + w2 * holdout_probs["p_thermal_risk"]
    + w3 * holdout_probs["p_plantar_risk"]
)

y_true = holdout_probs["fused_label"].values

evaluate(y_true, holdout_probs["p_rgb"].values, "RGB only (baseline)")
evaluate(y_true, holdout_probs["p_thermal_risk"].values, "Thermal only (baseline)")
evaluate(y_true, holdout_probs["p_plantar_risk"].values, "Plantar only (baseline)")
evaluate(y_true, holdout_probs["p_fused"].values, "Fused (equal weights, placeholder)")


--- RGB only (baseline) ---
Accuracy: 94.65% | AUC: 0.9697
              precision    recall  f1-score   support

      No DFU       0.95      0.74      0.83        54
         DFU       0.95      0.99      0.97       245

    accuracy                           0.95       299
   macro avg       0.95      0.87      0.90       299
weighted avg       0.95      0.95      0.94       299


--- Thermal only (baseline) ---
Accuracy: 78.93% | AUC: 0.7317
              precision    recall  f1-score   support

      No DFU       0.44      0.65      0.53        54
         DFU       0.91      0.82      0.86       245

    accuracy                           0.79       299
   macro avg       0.68      0.73      0.70       299
weighted avg       0.83      0.79      0.80       299


--- Plantar only (baseline) ---
Accuracy: 76.25% | AUC: 0.6820
              precision    recall  f1-score   support

      No DFU       0.40      0.63      0.49        54
         DFU       0.91      0.79      0.85      

(0.8762541806020067, np.float64(0.8845804988662131))